# Exercise 1: Web Server Access Log Analysis
**Student Name:** Solis  
**Student ID:** `TUPM-26-2631`  

### Mission
Develop a file-based log analyzer that processes a student-specific number of web requests, identifies one HTTP response class, and archives the findings.

### Data Generation Rules
- **Records Processed:** First 1000 + ID_SUM data records (`1000 + 20 = 1020`)
- **Target Status Class:** `(SEED_DIGIT % 4) + 2` = `(1 % 4) + 2 = 3` -> **3xx** (300-399)
- **Log Fields:** `IP`, `Time`, `URL`, `Staus` (dataset column name)


### Step 1: Student Information & Parameter Derivation


In [1]:
LAST_NAME = "Solis"
STUDENT_ID = "TUPM-26-2631"
SEED_DIGIT = int(STUDENT_ID[-1])  # 1
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())  # 20

RECORDS_LIMIT = 1000 + ID_SUM  # 1020
TARGET_CLASS_DIGIT = (SEED_DIGIT % 4) + 2  # 3 -> 3xx
TARGET_STATUS_CLASS = f"{TARGET_CLASS_DIGIT}xx"

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to Process (1000 + ID_SUM):", RECORDS_LIMIT)
print("Target Status Class ((SEED_DIGIT % 4) + 2):", TARGET_STATUS_CLASS)


Student: Solis
Student ID: TUPM-26-2631
Seed Digit: 1
ID Sum: 20
Records to Process (1000 + ID_SUM): 1020
Target Status Class ((SEED_DIGIT % 4) + 2): 3xx


### Step 2: Directory & File Path Setup


In [2]:
import os
import csv
import json
import shutil
from collections import Counter
from pathlib import Path

BASE_DIR = Path(".")
DATASET_PATH = BASE_DIR / "datasets" / "weblog.csv"
if not DATASET_PATH.exists():
    DATASET_PATH = BASE_DIR.parent / "datasets" / "weblog.csv"

OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_CSV_PATH = OUTPUT_DIR / "weblog_3xx_filtered.csv"
JSON_SUMMARY_PATH = OUTPUT_DIR / "weblog_3xx_summary.json"
BACKUP_JSON_PATH = BACKUP_DIR / "weblog_3xx_summary_backup.json"

print("Dataset Path:", DATASET_PATH.name)
print("Dataset exists:", DATASET_PATH.exists())
print("Output Directory exists:", OUTPUT_DIR.exists())
print("Backup Directory exists:", BACKUP_DIR.exists())


Dataset Path: weblog.csv
Dataset exists: True
Output Directory exists: True
Backup Directory exists: True


### Step 3: Read CSV Line by Line (Without Loading Entire Dataset into Memory)
*Extracts IP address, timestamp, requested URL, and HTTP status code.*


In [3]:
raw_records = []
records_processed = 0
invalid_records = 0

with open(DATASET_PATH, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    print("Dataset Columns:", reader.fieldnames)
    for row in reader:
        if records_processed >= RECORDS_LIMIT:
            break
        records_processed += 1
        
        ip = (row.get("IP") or "").strip()
        time = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status = (row.get("Staus") or row.get("Status") or "").strip()
        
        if not status or not status.isdigit():
            invalid_records += 1
            continue
            
        raw_records.append({
            "IP": ip,
            "Time": time,
            "URL": url,
            "Status": status
        })

print(f"Total records processed: {records_processed}")
print(f"Valid extracted records: {len(raw_records)}")
print(f"Invalid or skipped records: {invalid_records}")


Dataset Columns: ['IP', 'Time', 'URL', 'Staus']
Total records processed: 1020
Valid extracted records: 1020
Invalid or skipped records: 0


### Step 4: Select Records in Target Status Class (3xx)


In [4]:
selected_records = [
    r for r in raw_records 
    if r["Status"].startswith(str(TARGET_CLASS_DIGIT)) and len(r["Status"]) == 3
]

records_in_target = len(selected_records)
print(f"Target status class: {TARGET_STATUS_CLASS}")
print(f"Records matching target class: {records_in_target}")
print("\nSample matching record:")
print(selected_records[0])


Target status class: 3xx
Records matching target class: 164

Sample matching record:
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:02', 'URL': 'POST /process.php HTTP/1.1', 'Status': '302'}


### Step 5: Count Requests per Status Code & Top 5 Requested URLs


In [5]:
status_counts = Counter(r["Status"] for r in selected_records)
url_counts = Counter(r["URL"] for r in selected_records)
top_5_urls = url_counts.most_common(5)

print("Requests per Status Code:")
for code, cnt in sorted(status_counts.items()):
    print(f"  Status {code}: {cnt} requests")

print("\nFive Most Frequently Requested URLs:")
for i, (url, cnt) in enumerate(top_5_urls, 1):
    print(f"  {i}. {url} ({cnt} requests)")


Requests per Status Code:
  Status 302: 143 requests
  Status 304: 21 requests

Five Most Frequently Requested URLs:
  1. GET / HTTP/1.1 (45 requests)
  2. POST /process.php HTTP/1.1 (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1 (9 requests)
  5. GET /logout.php HTTP/1.1 (7 requests)


### Step 6: Export Selected Records to CSV and Write Summary to JSON


In [6]:
fieldnames = ["IP", "Time", "URL", "Status"]

with open(FILTERED_CSV_PATH, mode="w", encoding="utf-8", newline="") as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_records)

with open(FILTERED_CSV_PATH, mode="r", encoding="utf-8") as cf:
    exported_row_count = sum(1 for _ in csv.DictReader(cf))

summary_data = {
    "assessment_data": {
        "student_name": LAST_NAME,
        "student_id": STUDENT_ID,
        "seed_digit": SEED_DIGIT,
        "id_sum": ID_SUM,
        "records_processed": records_processed,
        "target_status_class": TARGET_STATUS_CLASS,
        "records_in_target_class": records_in_target,
        "invalid_or_skipped_records": invalid_records,
        "count_per_status_code": dict(status_counts),
        "five_most_requested_urls": [
            {"rank": i + 1, "url": u, "count": c}
            for i, (u, c) in enumerate(top_5_urls)
        ],
        "filtered_csv_file": FILTERED_CSV_PATH.name,
        "json_summary_file": JSON_SUMMARY_PATH.name
    }
}

with open(JSON_SUMMARY_PATH, mode="w", encoding="utf-8") as jf:
    json.dump(summary_data, jf, indent=4)

print(f"Exported CSV: {FILTERED_CSV_PATH.name} ({exported_row_count} data rows)")
print(f"Exported JSON: {JSON_SUMMARY_PATH.name} ({JSON_SUMMARY_PATH.stat().st_size} bytes)")


Exported CSV: weblog_3xx_filtered.csv (164 data rows)
Exported JSON: weblog_3xx_summary.json (1316 bytes)


### Step 7: Create Backup of JSON File & Verify It Exists


In [7]:
shutil.copy2(JSON_SUMMARY_PATH, BACKUP_JSON_PATH)

backup_exists = BACKUP_JSON_PATH.exists()
backup_size = BACKUP_JSON_PATH.stat().st_size if backup_exists else 0
backup_verified = backup_exists and backup_size > 0

print("Backup File Path:", BACKUP_JSON_PATH.name)
print("Backup File Exists:", backup_exists)
print(f"Backup File Size: {backup_size} bytes")
print("Backup Verification Status:", "VERIFIED" if backup_verified else "FAILED")


Backup File Path: weblog_3xx_summary_backup.json
Backup File Exists: True
Backup File Size: 1316 bytes
Backup Verification Status: VERIFIED


### Step 8: Assessment Data Table & Final Verification


In [8]:
print("=" * 70)
print(f"{'Item':<35} | {'Result'}")
print("-" * 70)
print(f"{'Records processed':<35} | {records_processed}")
print(f"{'Target status class':<35} | {TARGET_STATUS_CLASS}")
print(f"{'Records in target class':<35} | {records_in_target}")
status_str = ", ".join(f"{k}: {v}" for k, v in sorted(status_counts.items()))
print(f"{'Count per status code':<35} | {status_str}")
print(f"{'Five most requested URLs':<35} |")
for i, (u, c) in enumerate(top_5_urls, 1):
    print(f"  {i}. {u:<50} ({c} requests)")
print(f"{'Filtered CSV file':<35} | {FILTERED_CSV_PATH.name} ({len(selected_records)} rows)")
print(f"{'JSON summary file':<35} | {JSON_SUMMARY_PATH.name}")
print(f"{'Backup verification':<35} | {'VERIFIED' if backup_verified else 'FAILED'} ({BACKUP_JSON_PATH.name}, {backup_size} bytes)")
print("=" * 70)
print("\nFinal Output: Exercise 1 completed successfully!")


Item                                | Result
----------------------------------------------------------------------
Records processed                   | 1020
Target status class                 | 3xx
Records in target class             | 164
Count per status code               | 302: 143, 304: 21
Five most requested URLs            |
  1. GET / HTTP/1.1                                     (45 requests)
  2. POST /process.php HTTP/1.1                         (35 requests)
  3. GET /contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest HTTP/1.1 (34 requests)
  4. GET /home.php HTTP/1.1                             (9 requests)
  5. GET /logout.php HTTP/1.1                           (7 requests)
Filtered CSV file                   | weblog_3xx_filtered.csv (164 rows)
JSON summary file                   | weblog_3xx_summary.json
Backup verification                 | VERIFIED (weblog_3xx_summary_backup.json, 1316 bytes)

Final Output: Exercise 1 completed successfully!
